# 单节点多中断串行
需要启动三次，每次输入结果后会进入下一次中断

第二次中断恢复输入，触发第三次中断=》三个中断都在 get_info_node节点中，没次恢复都会执行整个函数，由于resume
中已经存储(存放在checkpoint中)了username信息，第二触发中断时 会判断检查点是否已经包含对应resume信息，检查到有不用再次输入

In [1]:

from typing import TypedDict, Literal

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import interrupt, Command


# 1、定义节点
class OverAllState(TypedDict):
    username: str
    age: int
    gender: Literal["male", "female"]


# 2、定义节点
def get_info_node(state: OverAllState) -> OverAllState:
    username = interrupt("请输入您的用户名：")
    age = interrupt("请输入您的年龄：")
    gender = interrupt("请输入您的性别:(male/female)")

    return {
        "username": username,
        "age": age,
        "gender": gender
    }


# 3、构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("get_info_node", get_info_node)
builder.add_edge(START, "get_info_node")
builder.add_edge("get_info_node", END)

# 4、创建检查点后端
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

# 触发中断
config = {"configurable": {"thread_id": "seq_interrupt_test"}}
username_interrupted_res = graph.invoke({}, config=config)
print('=' * 30, '-> username_interrupted_res <-', '=' * 30)
print(username_interrupted_res)

# 第一次恢复中断，输入姓名，同时触发第二次中断
user_name = input("请输入您的姓名:")
age_interrupted_res = graph.invoke(Command(resume=user_name), config=config)
print('=' * 30, '-> age_interrupted_res <-', '=' * 30)
print(age_interrupted_res)

# 第二次中断恢复输入，触发第三次中断=》三个中断都在 get_info_node节点中，没次恢复都会执行整个函数，由于resume 中已经存储了username信息，第二触发中断时不会要求再次输入usernaem
user_age = input("请输入您的年龄:")
gender_interrupted_res = graph.invoke(Command(resume=int(user_age)), config=config)
print('=' * 30, '-> age_interrupted_res <-', '=' * 30)
print(gender_interrupted_res)

# 第三次中断输入
user_gender = input("请输入您的性别: (male/female):")
resumed_res = graph.invoke(Command(resume=user_gender), config=config)
print('=' * 30, '-> age_interrupted_res <-', '=' * 30)
print(resumed_res)

============================== -> username_interrupted_res <- ==============================
{'__interrupt__': [Interrupt(value='请输入您的用户名：', id='3b648ffe6ec4a3c7dffda41871feb9f0')]}
============================== -> age_interrupted_res <- ==============================
{'__interrupt__': [Interrupt(value='请输入您的年龄：', id='3b648ffe6ec4a3c7dffda41871feb9f0')]}
============================== -> age_interrupted_res <- ==============================
{'__interrupt__': [Interrupt(value='请输入您的性别:(male/female)', id='3b648ffe6ec4a3c7dffda41871feb9f0')]}
============================== -> age_interrupted_res <- ==============================
{'username': 'sevan', 'age': 20, 'gender': 'male'}
